# Chapter 3 Practical 05: Evaluation with MAE, Precision@K, and Recall@K

Learning objectives:
- Create a leave-one-out test split.
- Evaluate rating prediction with MAE.
- Evaluate Top-K recommendation with Precision@K, Recall@K, and HitRate@K.
- Compare user-user and item-item CF against a popularity baseline.

Slide connection: practical exercise evaluation requirements and performance comparison.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Loaded ratings_chapter3.csv from data/ratings_chapter3.csv
Loaded movies_chapter3.csv from data/movies_chapter3.csv


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
user_id,,,,,,,,,,
Alice,5.0,NaN,4.0,NaN,4.0,NaN,5.0,NaN,NaN,NaN
Bob,NaN,NaN,6.0,4.0,7.0,4.0,7.0,NaN,NaN,NaN
Chris,NaN,NaN,2.0,7.0,3.0,7.0,NaN,NaN,NaN,5.0
Karen,NaN,NaN,NaN,4.0,7.0,3.0,6.0,NaN,NaN,NaN
Lynn,NaN,NaN,2.0,4.0,4.0,6.0,NaN,NaN,NaN,6.0
Nina,NaN,5.0,NaN,4.0,NaN,NaN,NaN,NaN,2.0,5.0
Omar,NaN,2.0,NaN,3.0,NaN,NaN,NaN,5.0,5.0,NaN
Sally,NaN,NaN,7.0,6.0,7.0,3.0,6.0,NaN,NaN,NaN


## Packages and key functions used

- `pandas` is used for tabular data. Important functions in this notebook include `read_csv`, `merge`, `pivot_table`, `groupby`, `agg`, `sort_values`, and `dropna`.
- `numpy` is used for numerical operations such as vector norms, dot products, averages, absolute values, and exponential time-decay weights.
- `pathlib.Path` helps the notebook find local CSV files in Colab, Jupyter, or the repository folder.
- The shared helper `read_chapter3_csv()` first looks for local data files and then falls back to the GitHub raw URL when students open the notebook directly online.
- `rating_matrix` is the central memory-based collaborative filtering object: rows are users, columns are movies, observed numbers are ratings, and missing cells mean unknown preferences.


## Technique: leave-one-out evaluation split

This notebook removes one rating per user as a small test set. The remaining ratings form `train_matrix`, which is used for prediction and recommendation. This keeps the evaluation simple: each user has one hidden item that the recommender tries to recover or predict.


In [2]:
np.random.seed(7)
test_rows = ratings_named.groupby("user_id", group_keys=False).sample(n=1, random_state=7)
train_rows = ratings_named.drop(test_rows.index)
train_matrix = train_rows.pivot_table(index="user_id", columns="title", values="rating")

test_rows[["user_id", "title", "rating"]].sort_values("user_id")


,user_id,title,rating
26,Alice,Star Wars,4
7,Bob,Terminator 2,4
13,Chris,Independence Day,2
23,Karen,The Matrix,6
16,Lynn,Jurassic Park,4
29,Nina,Finding Nemo,5
33,Omar,The Notebook,5
2,Sally,Terminator 2,3


## Technique: prediction functions for evaluation

The next cell defines reusable prediction helpers:

- `predict_user_user_eval()` predicts a hidden rating from similar users.
- `build_item_sim()` creates item-item similarities from the training matrix.
- `predict_item_item_eval()` predicts a hidden rating from similar items the user already rated.

These functions are intentionally compact so students can focus on how evaluation calls them.


In [3]:
def pearson_on_overlap(matrix, user_a, user_b):
    if user_a not in matrix.index or user_b not in matrix.index:
        return np.nan
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    if pair.shape[1] < 2:
        return np.nan
    if pair.loc[user_a].std() == 0 or pair.loc[user_b].std() == 0:
        return np.nan
    return float(np.corrcoef(pair.loc[user_a], pair.loc[user_b])[0, 1])

def predict_user_user_eval(matrix, target_user, item, k=3):
    if target_user not in matrix.index or item not in matrix.columns:
        return np.nan
    rows = []
    for other in matrix.index.drop(target_user):
        if pd.isna(matrix.loc[other, item]):
            continue
        sim = pearson_on_overlap(matrix, target_user, other)
        if not pd.isna(sim) and sim > 0:
            rows.append((other, sim))
    rows = sorted(rows, key=lambda x: x[1], reverse=True)[:k]
    if not rows:
        return np.nan
    target_mean = matrix.loc[target_user].mean()
    numerator = sum(sim * (matrix.loc[u, item] - matrix.loc[u].mean()) for u, sim in rows)
    denominator = sum(abs(sim) for _, sim in rows)
    return target_mean + numerator / denominator if denominator else np.nan

def build_item_sim(matrix):
    sim = pd.DataFrame(index=matrix.columns, columns=matrix.columns, dtype=float)
    for a in matrix.columns:
        for b in matrix.columns:
            pair = matrix[[a, b]].dropna()
            sim.loc[a, b] = 1.0 if a == b else (
                np.corrcoef(pair[a], pair[b])[0, 1]
                if len(pair) >= 2 and pair[a].std() != 0 and pair[b].std() != 0
                else np.nan
            )
    return sim

train_item_sim = build_item_sim(train_matrix)

def predict_item_item_eval(matrix, user, item, k=3):
    if user not in matrix.index or item not in train_item_sim.index:
        return np.nan
    rated = matrix.loc[user].dropna()
    rows = []
    for rated_item, rating in rated.items():
        sim = train_item_sim.loc[item, rated_item]
        if not pd.isna(sim) and sim > 0:
            rows.append((rated_item, rating, sim))
    rows = sorted(rows, key=lambda x: x[2], reverse=True)[:k]
    if not rows:
        return np.nan
    numerator = sum(rating * sim for _, rating, sim in rows)
    denominator = sum(abs(sim) for _, _, sim in rows)
    return numerator / denominator if denominator else np.nan

print("Evaluation helper functions ready: predict_user_user_eval, build_item_sim, predict_item_item_eval")


Evaluation helper functions ready: predict_user_user_eval, build_item_sim, predict_item_item_eval


## Technique: MAE for rating prediction

Mean Absolute Error (MAE) compares predicted ratings with the hidden actual ratings. Lower MAE means the predicted ratings are closer to the true held-out ratings.


In [4]:
def popularity_prediction(matrix, item):
    return matrix[item].mean() if item in matrix.columns else matrix.stack().mean()

pred_rows = []
for _, row in test_rows.iterrows():
    user, item, actual = row["user_id"], row["title"], row["rating"]
    for model, pred in [
        ("user_user", predict_user_user_eval(train_matrix, user, item)),
        ("item_item", predict_item_item_eval(train_matrix, user, item)),
        ("popularity", popularity_prediction(train_matrix, item)),
    ]:
        if not pd.isna(pred):
            pred_rows.append({"model": model, "user": user, "item": item, "actual": actual, "predicted": pred})

predictions = pd.DataFrame(pred_rows)
predictions["absolute_error"] = (predictions["actual"] - predictions["predicted"]).abs()
predictions.groupby("model")["absolute_error"].mean().rename("MAE").round(3)


model
item_item     1.204
popularity    1.465
user_user     1.271
Name: MAE, dtype: float64

## Technique: Precision@K, Recall@K, and HitRate@K

Top-k metrics evaluate the ranked recommendation list rather than a single rating prediction.

- `topk_popularity()` recommends unseen movies with the highest average rating.
- `topk_user_user()` recommends unseen movies with the highest predicted user-user score.
- `evaluate_topk()` compares the recommended list with held-out relevant items.
- Precision@K asks: out of K recommended items, how many were relevant?
- Recall@K asks: out of all relevant hidden items, how many did we recover?
- HitRate@K asks: did we get at least one relevant item?


In [5]:
def topk_popularity(matrix, user, k=3):
    seen = set(matrix.loc[user].dropna().index)
    scores = matrix.mean().drop(labels=list(seen), errors="ignore")
    return scores.sort_values(ascending=False).head(k).index.tolist()

def topk_user_user(matrix, user, k=3):
    unseen = matrix.columns[matrix.loc[user].isna()]
    rows = [(item, predict_user_user_eval(matrix, user, item)) for item in unseen]
    rows = [(item, score) for item, score in rows if not pd.isna(score)]
    return [item for item, _ in sorted(rows, key=lambda x: x[1], reverse=True)[:k]]

def evaluate_topk(model_fn, train_matrix, test_rows, k=3, relevant_threshold=5):
    rows = []
    for user, user_test in test_rows.groupby("user_id"):
        relevant = set(user_test.loc[user_test["rating"] >= relevant_threshold, "title"])
        recommended = model_fn(train_matrix, user, k)
        hits = len(set(recommended) & relevant)
        rows.append({
            "user": user,
            "precision_at_k": hits / k,
            "recall_at_k": hits / len(relevant) if relevant else np.nan,
            "hit_rate_at_k": 1 if hits > 0 else 0,
            "recommended": recommended,
            "relevant": sorted(relevant),
        })
    return pd.DataFrame(rows)

pop_eval = evaluate_topk(topk_popularity, train_matrix, test_rows)
uu_eval = evaluate_topk(topk_user_user, train_matrix, test_rows)

pd.DataFrame({
    "model": ["popularity", "user_user"],
    "precision_at_3": [pop_eval["precision_at_k"].mean(), uu_eval["precision_at_k"].mean()],
    "recall_at_3": [pop_eval["recall_at_k"].mean(), uu_eval["recall_at_k"].mean()],
    "hit_rate_at_3": [pop_eval["hit_rate_at_k"].mean(), uu_eval["hit_rate_at_k"].mean()],
}).round(3)


,model,precision_at_3,recall_at_3,hit_rate_at_3
0,popularity,0.042,0.333,0.125
1,user_user,0.042,0.333,0.125


# Challenges

### Challenge 1 — Change the Relevance Threshold

**Goal:**
Investigate how the definition of a relevant item changes Precision@K and Recall@K.

**What to do:**

1. In `evaluate_topk`, change `relevant_threshold` from `5` to `6`.
2. Rerun the popularity and user-user evaluation cells.
3. Compare Precision@3, Recall@3, and HitRate@3 with the original result.
4. Explain why stricter relevance can change the metrics.


In [6]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Which metric changed most? Did both models change in the same way? What does a stricter relevance threshold mean for evaluation?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 — Change the Recommendation List Length

**Goal:**
Investigate how evaluating a longer recommendation list changes top-k metrics.

**What to do:**

1. Change the evaluation value of `k` from `3` to `5`.
2. Rerun `evaluate_topk` for the popularity and user-user recommenders.
3. Update the summary table labels so they describe `Precision@5`, `Recall@5`, and `HitRate@5`.
4. Compare the new metrics with the original @3 result.


In [7]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Did Precision decrease or increase when `k` became larger? Did Recall change differently from Precision? Why can this happen?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 — Concept Check: Rating Error vs Top-K Quality

This challenge requires **no programming**.

A model has a low MAE, but its top-3 recommendation list contains no relevant items for a user.

Explain why rating-prediction accuracy and recommendation-list quality are related but not the same thing.

### Your explanation

> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
